# Homework 10 - ST 554

Author: Max Campbell

## Part 1 - Creating Streaming Data Using `rate`

In this assignment, we are tasked with creating data streaming processes to allow us to read in data over time. This is useful in big data contexts where we often need to read in large amounts of data at a high velocity. Luckily for us, Spark allows us to stream data! We can create testing data using the `rate` format to demonstrate the basics of this process. First, let's initialize the Spark session and the corresponding stream reader:

In [1]:
#Load necessary modules
from pyspark.sql import SparkSession
from pyspark.sql.functions import sqrt, try_mod, col, lit


#Initialize SparkSession
spark = SparkSession.builder.master('local[*]').appName('HW9') \
    .config("spark.sql.ansi.enabled", "false").getOrCreate()

#Set up the stream
df = spark \
    .readStream \
    .format("rate") \
    .option("rowsPerSecond", 1) \
    .load()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 14:42:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


As with any dataset, we want to be able to perform operations on the observations we collect. We will demonstrate how to do this with Spark by finding the square root and modulus 4 of the value of interest in our `rate` data.

In [2]:
#Set up computations that we want to perform on the stream
#Note: lit function casts 4 to a Column object.
query = df.withColumn('sqrt', sqrt(col('value'))).withColumn('mod_4', try_mod(col('value'), lit(4)))

The last thing to do is start the stream! We'll let it run for about 30 seconds, having it write to the dataframe in-memory, and then (manually) stop it.

In [5]:
#Start stream, having it write to memory
current_stream = query.writeStream.format("memory").queryName("rateTable").start()

26/04/17 14:42:28 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-3fae7eec-25f0-4320-b316-d3e4b3e11e58. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/17 14:42:28 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [6]:
#Stop output
current_stream.stop()

#Stop stream, and view the output
spark.sql("select * from rateTable").show()

26/04/17 14:43:13 WARN DAGScheduler: Failed to cancel job group fff03e17-67c8-4bad-8ff0-20f93bf08875. Cannot find active jobs for it.
26/04/17 14:43:13 WARN DAGScheduler: Failed to cancel job group fff03e17-67c8-4bad-8ff0-20f93bf08875. Cannot find active jobs for it.


+--------------------+-----+------------------+-----+
|           timestamp|value|              sqrt|mod_4|
+--------------------+-----+------------------+-----+
|2026-04-17 14:42:...|    0|               0.0|    0|
|2026-04-17 14:42:...|    1|               1.0|    1|
|2026-04-17 14:42:...|    2|1.4142135623730951|    2|
|2026-04-17 14:42:...|    3|1.7320508075688772|    3|
|2026-04-17 14:42:...|    4|               2.0|    0|
|2026-04-17 14:42:...|    5|  2.23606797749979|    1|
|2026-04-17 14:42:...|    6| 2.449489742783178|    2|
|2026-04-17 14:42:...|    7|2.6457513110645907|    3|
|2026-04-17 14:42:...|    8|2.8284271247461903|    0|
|2026-04-17 14:42:...|    9|               3.0|    1|
|2026-04-17 14:42:...|   10|3.1622776601683795|    2|
|2026-04-17 14:42:...|   11|   3.3166247903554|    3|
|2026-04-17 14:42:...|   12|3.4641016151377544|    0|
|2026-04-17 14:42:...|   13| 3.605551275463989|    1|
|2026-04-17 14:42:...|   14|3.7416573867739413|    2|
|2026-04-17 14:42:...|   15|

We've successfully streamed some data, found the square root and the mod 4 for the `value` column, and saved it in-memory!